# 04 — Explore the charts

**The question (F4):** what is *the shape of game review scores*? Using **IGDB critic ratings** (switched from Metacritic so recent years aren't starved — IGDB's critic coverage keeps up). 6,005 games with a critic aggregate backed by ≥3 outlets.

**Headline facts (`03-prepare`):** mean 73.5, median 75.3; the 90+ club is only ~246 games (~4%); the 70s are the fat middle (~2,100); sub-50 is rare (~260, ~4%). A **left-skewed pile in the 70s–80s with a thin elite tail and a surprisingly thin bad-score tail**.

Three charts explored below (display-only). **⭐ Review surface — confirm the framing before I build the polished social/web renders in `06-viz-social`.**

*Rendering note: this venv (Py3.14 + matplotlib) hits the known RecursionError on matplotlib ticks, so exploration uses the shared Pillow templates — the real publication charts — rendered inline.*

In [ ]:
import sys, os
from pathlib import Path

PROJECT = Path.cwd()
while not (PROJECT / 'config.yaml').exists() and PROJECT != PROJECT.parent:
    PROJECT = PROJECT.parent
os.chdir(PROJECT)
sys.path.insert(0, str(PROJECT))
SHARED = PROJECT.parent.parent / 'shared'
sys.path.insert(0, str(SHARED))

import duckdb
from colors import c
from chart_templates import histogram, line_chart, scatter_plot
from IPython.display import display

from src.ingest import load_config
cfg = load_config('config.yaml')
con = duckdb.connect(cfg['settings']['duckdb_file'], read_only=True)

df_all = con.execute('SELECT critic_rating, user_rating, release_year FROM chart_critic_all').df()
print('games:', len(df_all))
df_all['critic_rating'].describe()

## Chart 1 (HERO) — distribution of IGDB critic ratings

Full 0–100 range, 5-point bands, mean + median lines, and the **90+ club highlighted in aqua** (deliberately NOT gold — the gold dashed line is the median, and we don't want the highlight to read as related to it). This is the F4 hero: "the shape of game review scores."

In [ ]:
img = histogram(
    df_all, value_col='critic_rating',
    bin_edges=list(range(0, 101, 5)), x_range=(0, 100), x_tick_step=10,
    x_axis_label='IGDB critic rating', y_axis_label='Number of games',
    bar_color=c('teal'), highlight_range=(90, 100, c('aqua')),
    title='Video game review scores — distribution of IGDB critic ratings',
    subtitle='6,005 games (≥ 3 critic scores each); each bar = a 5-point band. 90+ highlighted.',
    source='IGDB',
)
display(img)

## Chart 2 — games scoring 90+ per year

The elite tail over time. Cut at the last complete year (recent releases are still accruing critic scores, so a partial year would look like a false collapse) — the cutoff + reason are stated in the subtitle.

In [ ]:
by_year = con.execute('SELECT year, n_90plus FROM chart_90plus_by_year ORDER BY year').df()
cut = int(by_year['year'].max())
print('spans', int(by_year['year'].min()), '→', cut)
img = line_chart(
    by_year, x_col='year',
    series=[{'col': 'n_90plus', 'label': '90+ games', 'color': c('teal')}],
    x_axis_label='Release year', y_axis_label='Games scoring 90+',
    y_min=0, value_labels=True, label_last=False, markers=True,
    title='Video game review scores — games scoring 90+ by release year',
    subtitle=f'IGDB critic rating. Cut at {cut}: newer releases are still accruing critic scores, so later years would undercount.',
    source='IGDB',
)
display(img)

## Chart 3 — user rating vs critic rating

Do IGDB users agree with critics? Point per game (the ~5,300 with both a critic and a user rating). Look for the correlation and where they diverge.

In [ ]:
df_scatter = con.execute('SELECT critic_rating, user_rating FROM chart_critic_all WHERE user_rating IS NOT NULL').df()
print('games with both ratings:', len(df_scatter))
corr = df_scatter['critic_rating'].corr(df_scatter['user_rating'])
print('Pearson r:', round(corr, 3))
img = scatter_plot(
    df_scatter, x_col='critic_rating', y_col='user_rating',
    x_axis_label='IGDB critic rating', y_axis_label='IGDB user rating',
    point_color=c('teal'),
    title='Video game review scores — IGDB user rating vs critic rating',
    subtitle=f'{len(df_scatter):,} games with both a critic and a user rating (0–100). Pearson r = {corr:.2f}.',
    source='IGDB',
)
display(img)

## Score-band table (caption numbers)

In [ ]:
con.execute('''
SELECT
  CASE
    WHEN critic_rating >= 90 THEN '90-100 (elite)'
    WHEN critic_rating >= 80 THEN '80-89 (great)'
    WHEN critic_rating >= 70 THEN '70-79 (good)'
    WHEN critic_rating >= 60 THEN '60-69 (mixed)'
    WHEN critic_rating >= 50 THEN '50-59 (weak)'
    ELSE 'below 50 (bad)'
  END AS band,
  COUNT(*) AS n_games,
  ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 1) AS pct
FROM chart_critic_all
GROUP BY band ORDER BY MIN(critic_rating) DESC
''').df()

---
## Framing to confirm

1. **Hero histogram** (Chart 1) — full 0–100, 90+ in aqua, median gold. Good?
2. **90+ per year line** (Chart 2) — cutoff year + caption OK?
3. **User-vs-critic scatter** (Chart 3) — worth publishing? (Check the r value and whether the cloud tells a story — e.g. users rating high-critic games lower, or a tight agreement.)

Once confirmed, I'll build the finalized social + web renders in `06-viz-social`.

---
## Cleanup
Close the (read-only) DuckDB connection so the lock is released.

In [ ]:
con.close()
print('connection closed')